In [ ]:
import subprocess

print("Installing dependencies (skip mamba-ssm)...")
subprocess.run(
    "pip install timm scipy opencv-python-headless matplotlib "
    "huggingface_hub einops torch --quiet",
    shell=True
)

print("✅  Dependencies installed in <2 min")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("✅  GPU ready")

MAMBA_AVAILABLE = False  # Will handle this in VMambaBackbone

In [ ]:
# CELL 2

import os, math, random, warnings, time, pickle, shutil
from pathlib import Path

import cv2
import numpy as np
import scipy.io as sio
import scipy.ndimage
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import transforms

import timm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅  Imports done")
print(f"   Device : {DEVICE}")
if DEVICE.type == "cpu":
    print("   ⚠️  WARNING: No GPU detected — inference will be slow")


In [ ]:
# CELL 3

from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive", force_remount=False)
print("✅  Drive mounted")

GDRIVE_ROOT    = Path("/content/drive/MyDrive/Metro-Crowd-Project")
DATA_ROOT      = GDRIVE_ROOT / "datasets" / "ShanghaiTech"
CHECKPOINT_DIR = GDRIVE_ROOT / "checkpoints"
CACHE_DIR      = GDRIVE_ROOT / "density_cache"

for d in [CHECKPOINT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 2. Use dataset DIRECTLY from Drive (skip local copy)
LOCAL_DATA = DATA_ROOT  # Point directly to Drive instead of copying

print(f"✅  Dataset source: {LOCAL_DATA}")
print(f"    (Using directly from Drive — no local copy)")

# 3. Verify
assert LOCAL_DATA.exists(), "❌  Dataset not found on Drive"
print(f"✅  Dataset ready at {LOCAL_DATA}")

In [ ]:
# CELL 4

# ════════════════════════════════════════════════════════════════
#  EDIT HERE — all config in one place
# ════════════════════════════════════════════════════════════════

# ── Dataset ───────────────────────────────────────────────────
DATASET_PART = "part_A"          # "part_B" (avg ~123 heads) | "part_A" (avg ~300 heads)

# ── Paths (Drive) ─────────────────────────────────────────────
GDRIVE_ROOT    = Path("/content/drive/MyDrive/Metro-Crowd-Project")
CHECKPOINT_DIR = GDRIVE_ROOT / "checkpoints"
CACHE_DIR      = GDRIVE_ROOT / "density_cache"
CHECKPOINT_PATH = CHECKPOINT_DIR / f"vmamba_crowd_{DATASET_PART}_best.pth"
TRAIN_CACHE    = CACHE_DIR / f"train_density_cache_{DATASET_PART}.pkl"
TEST_CACHE     = CACHE_DIR / f"test_density_cache_{DATASET_PART}.pkl"
CURVES_PATH    = CHECKPOINT_DIR / f"vmamba_{DATASET_PART}_curves.png"

# ── Paths (Local — fast I/O during training) ──────────────────
LOCAL_DATA     = GDRIVE_ROOT / "datasets" / "ShanghaiTech"  # Point to Drive
TRAIN_IMG_PATH = LOCAL_DATA / DATASET_PART / "train_data" / "images"
TRAIN_GT_PATH  = LOCAL_DATA / DATASET_PART / "train_data" / "ground-truth"
TEST_IMG_PATH  = LOCAL_DATA / DATASET_PART / "test_data"  / "images"
TEST_GT_PATH   = LOCAL_DATA / DATASET_PART / "test_data"  / "ground-truth"

# ── Model ─────────────────────────────────────────────────────
DENSITY_SCALE   = 8    # output density map = H/8 × W/8
GAUSSIAN_SIGMA  = 15   # fixed-sigma Gaussian for GT density maps
FPN_CHANNELS    = 256  # FPN feature dimension

# ── Training ──────────────────────────────────────────────────
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS    = 150    # total epochs across all Colab sessions
WARMUP_EPOCHS = 10     # linear LR warmup
BATCH_SIZE    = 8      # fits comfortably on T4 with 256×256 crops
CROP_SIZE     = 256    # random crop size for training
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
LR_MIN        = 1e-6
LAMBDA_COUNT  = 0.1    # weight for count L1 loss term
NUM_WORKERS   = 0      # 0 = safe on Colab; increase if using persistent runtime
PIN_MEMORY    = False  # False = safe on Colab shared memory
SEED          = 42

# ── ImageNet normalisation ─────────────────────────────────────
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

# ── Reproducibility ───────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Ensure dirs exist ─────────────────────────────────────────
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Verify dataset paths ──────────────────────────────────────
print(f"Dataset      : ShanghaiTech {DATASET_PART}")
print(f"Device       : {DEVICE}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
for name, p in [("Train Img", TRAIN_IMG_PATH), ("Train GT",  TRAIN_GT_PATH),
                ("Test Img",  TEST_IMG_PATH),  ("Test GT",   TEST_GT_PATH)]:
    status = "✅" if p.exists() else "❌"
    count  = len(list(p.glob("*"))) if p.exists() else 0
    print(f"  {status} {name}: {p.name}  ({count} files)")

# ── Resume from checkpoint ───────────────────────────────────
RESUME = False

In [ ]:
# CELL 5

def load_gt_points(mat_path: str) -> np.ndarray:
    """Load (x, y) head annotation points from a ShanghaiTech .mat file."""
    mat = sio.loadmat(mat_path)
    pts = mat["image_info"][0][0][0][0][0]
    return pts.astype(np.float32)   # shape: (N, 2)


def generate_density_map(image_shape: tuple, points: np.ndarray,
                          sigma: int = GAUSSIAN_SIGMA) -> np.ndarray:
    """
    Create a full-resolution Gaussian density map.
    Placing all dots at once then applying a single Gaussian filter is
    numerically equivalent to summing per-dot Gaussians but ~10× faster.
    """,
    H, W    = image_shape
    density = np.zeros((H, W), dtype=np.float32)
    if len(points) == 0:
        return density
    xs = np.clip(points[:, 0].astype(int), 0, W - 1)
    ys = np.clip(points[:, 1].astype(int), 0, H - 1)
    np.add.at(density, (ys, xs), 1.0)
    density = scipy.ndimage.gaussian_filter(density, sigma=sigma)
    return density


def precompute_density_maps(img_dir: Path, gt_dir: Path, cache_path: Path) -> None:
    """Precompute and cache all density maps for a split.  Idempotent."""
    cache_path = Path(cache_path)
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            existing = pickle.load(f)
        if len(existing) > 0:
            print(f"  ✅  Cache exists ({len(existing)} maps) — skipping: {cache_path.name}")
            return
        cache_path.unlink()  # empty cache — regenerate

    print(f"  Generating density maps from: {img_dir}")
    cache     = {}
    img_paths = sorted(Path(img_dir).glob("*.jpg"))
    total     = len(img_paths)
    t0        = time.time()

    for i, img_path in enumerate(img_paths):
        gt_path = Path(gt_dir) / f"GT_{img_path.stem}.mat"
        if not gt_path.exists():
            print(f"    ⚠️  Missing: {gt_path.name}")
            continue
        img     = cv2.imread(str(img_path))
        H, W    = img.shape[:2]
        points  = load_gt_points(str(gt_path))
        density = generate_density_map((H, W), points)
        cache[img_path.stem] = density
        if (i + 1) % 50 == 0 or (i + 1) == total:
            elapsed = time.time() - t0
            eta     = (elapsed / (i + 1)) * (total - i - 1)
            print(f"    {i+1:4d}/{total}  ETA: {eta:.0f}s")

    with open(cache_path, "wb") as f:
        pickle.dump(cache, f)
    print(f"  ✅  Saved {len(cache)} maps → {cache_path}")


# Quick sanity-check on one image (does not use cache)
sample_gt  = next(TRAIN_GT_PATH.glob("*.mat"))
sample_img = TRAIN_IMG_PATH / (sample_gt.stem.replace("GT_", "") + ".jpg")
pts        = load_gt_points(str(sample_gt))
img_bgr    = cv2.imread(str(sample_img))
H, W       = img_bgr.shape[:2]
dmap       = generate_density_map((H, W), pts)
print(f"Sanity check: {sample_img.name}")
print(f"  GT heads    : {len(pts)}  |  density sum: {dmap.sum():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Image — {sample_img.name}")
axes[0].axis("off")
axes[1].imshow(dmap, cmap="jet")
axes[1].set_title(f"Density Map  (sum={dmap.sum():.1f},  heads={len(pts)})")
axes[1].axis("off")
plt.tight_layout(); plt.show()

# Precompute all density maps
print("\n=== Generating / loading density map caches ===")
print("Train:")
precompute_density_maps(TRAIN_IMG_PATH, TRAIN_GT_PATH, TRAIN_CACHE)
print("Test:")
precompute_density_maps(TEST_IMG_PATH,  TEST_GT_PATH,  TEST_CACHE)
print("\nDensity map caches ready.")


In [ ]:
# CELL 6

class ShanghaiTechDataset(Dataset):
    """
    ShanghaiTech crowd-counting dataset.

    Training mode: returns random (crop_size × crop_size) crops.
    Validation mode: returns the full image (crop_size=None).
    Density maps are read from a pre-computed pickle cache for speed.
    """

    def __init__(self, img_dir: Path, gt_dir: Path,
                 crop_size: int = None, augment: bool = False,
                 cache_path: Path = None):
        self.img_dir   = Path(img_dir)
        self.gt_dir    = Path(gt_dir)
        self.crop_size = crop_size
        self.augment   = augment
        self.normalize = transforms.Normalize(mean=IMG_MEAN, std=IMG_STD)

        self.img_paths = sorted(self.img_dir.glob("*.jpg"))
        if not self.img_paths:
            self.img_paths = sorted(self.img_dir.glob("*.png"))
        assert len(self.img_paths) > 0, f"No images found in {img_dir}"

        # Load density-map cache
        self.cache = None
        if cache_path and Path(cache_path).exists():
            with open(cache_path, "rb") as f:
                self.cache = pickle.load(f)
            print(f"  ✅  Loaded cache ({len(self.cache)} maps) from {Path(cache_path).name}")

    def __len__(self) -> int:
        return len(self.img_paths)

    def __getitem__(self, idx: int):
        img_path = self.img_paths[idx]
        img      = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        H, W     = img.shape[:2]

        # Retrieve density map
        if self.cache is not None:
            density = self.cache[img_path.stem].copy()
        else:
            gt_path = self.gt_dir / f"GT_{img_path.stem}.mat"
            points  = load_gt_points(str(gt_path))
            density = generate_density_map((H, W), points)

        # ── Random crop (training only) ──────────────────────────────
        if self.crop_size:
            cs = self.crop_size
            # Ensure the image is at least crop_size in both dimensions
            if H < cs or W < cs:
                scale   = cs / min(H, W) + 1e-3
                new_H   = max(cs, int(H * scale))
                new_W   = max(cs, int(W * scale))
                img     = cv2.resize(img,     (new_W, new_H))
                density = cv2.resize(density, (new_W, new_H),
                                     interpolation=cv2.INTER_LINEAR)
                H, W    = new_H, new_W
            x1 = random.randint(0, W - cs)
            y1 = random.randint(0, H - cs)
            img     = img    [y1:y1+cs, x1:x1+cs]
            density = density[y1:y1+cs, x1:x1+cs]

        # ── Augmentation ─────────────────────────────────────────────
        if self.augment:
            # Horizontal flip
            if random.random() > 0.5:
                img     = np.fliplr(img    ).copy()
                density = np.fliplr(density).copy()
            # Brightness / contrast jitter
            img = img.astype(np.float32)
            img *= random.uniform(0.7, 1.3)
            mu   = img.mean()
            img  = (img - mu) * random.uniform(0.8, 1.2) + mu
            img  = np.clip(img, 0, 255).astype(np.uint8)

        # ── Downscale density to model output resolution ──────────────
        dH = img.shape[0] // DENSITY_SCALE
        dW = img.shape[1] // DENSITY_SCALE
        count_before  = density.sum()
        density_small = cv2.resize(density, (dW, dH),
                                   interpolation=cv2.INTER_LINEAR)
        # Count-preserving normalisation
        if density_small.sum() > 1e-8:
            density_small = density_small * (count_before / density_small.sum())

        # ── Convert to tensors ────────────────────────────────────────
        img_t = self.normalize(
            torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        )
        den_t = torch.from_numpy(density_small).float().unsqueeze(0)
        return img_t, den_t, str(img_path)


def build_dataloaders():
    print("Building dataloaders...")
    train_ds = ShanghaiTechDataset(
        TRAIN_IMG_PATH, TRAIN_GT_PATH,
        crop_size=CROP_SIZE, augment=True,
        cache_path=TRAIN_CACHE)
    test_ds = ShanghaiTechDataset(
        TEST_IMG_PATH, TEST_GT_PATH,
        crop_size=None, augment=False,
        cache_path=TEST_CACHE)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    test_loader = DataLoader(
        test_ds, batch_size=1, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    print(f"  Train samples : {len(train_ds)}")
    print(f"  Test  samples : {len(test_ds)}")
    print(f"  Train batches : {len(train_loader)}  (batch_size={BATCH_SIZE})")
    return train_loader, test_loader


train_loader, test_loader = build_dataloaders()

# Batch sanity-check
imgs, densities, paths = next(iter(train_loader))
print(f"\nBatch check (training):")
print(f"  Image   : {list(imgs.shape)}  dtype={imgs.dtype}")
print(f"  Density : {list(densities.shape)}  dtype={densities.dtype}")
print(f"  Counts  : {[f'{c:.1f}' for c in densities.sum((1,2,3)).tolist()]}")


In [ ]:
# CELL 7

import importlib, subprocess, sys


# ── Helper: attempt to import VMamba from various sources ─────────────────────

def _try_hf_vmamba():
    """
    Primary approach: download VMamba-Small weights from HuggingFace Hub and
    load them into the model cloned from the official repo (HF hosts both).
    Returns (model_cls, ckpt_path) or raises ImportError.
    """
    from huggingface_hub import hf_hub_download
    import os

    # Ensure the VMamba source is available
    vmamba_src = Path("/content/VMamba")
    if not vmamba_src.exists():
        print("  Cloning VMamba from GitHub (depth=1)...")
        ret = subprocess.run(
            "git clone https://github.com/MzeroMiko/VMamba.git /content/VMamba --depth 1",
            shell=True, capture_output=True, text=True)
        if ret.returncode != 0:
            raise ImportError(f"git clone failed:\n{ret.stderr}")

    # Add source to path
    vmamba_cls_path = str(vmamba_src / "classification")
    if vmamba_cls_path not in sys.path:
        sys.path.insert(0, vmamba_cls_path)

    # Try to install VMamba's local dependencies if needed
    subprocess.run("pip install -e /content/VMamba --quiet",
                   shell=True, capture_output=True)

    # Download pretrained VMamba-Small weights
    print("  Downloading VMamba-Small ImageNet weights from HuggingFace Hub...")
    ckpt_path = hf_hub_download(
        repo_id="MzeroMiko/VMamba",
        filename="vssmsmall_dp03_ckpt_epoch_238.pth",
        cache_dir="/content/vmamba_weights",
    )
    print(f"  ✅  Weights cached at: {ckpt_path}")

    # Import VSSM (VMamba's backbone class)
    from models.vmamba import VSSM
    return VSSM, ckpt_path


def _try_github_vmamba():
    """
    Fallback: clone official repo, install, import VSSM, download weights manually.
    """
    vmamba_src = Path("/content/VMamba")
    if not vmamba_src.exists():
        print("  Cloning VMamba...")
        subprocess.run(
            "git clone https://github.com/MzeroMiko/VMamba.git /content/VMamba --depth 1",
            shell=True, check=True)
    subprocess.run("pip install -e /content/VMamba --quiet",
                   shell=True, capture_output=True)
    vmamba_cls_path = str(vmamba_src / "classification")
    if vmamba_cls_path not in sys.path:
        sys.path.insert(0, vmamba_cls_path)
    from models.vmamba import VSSM

    # Weights: try to download via wget from official release
    ckpt_path = "/content/vmamba_weights/vssmsmall_dp03_ckpt_epoch_238.pth"
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    if not os.path.exists(ckpt_path):
        url = ("https://github.com/MzeroMiko/VMamba/releases/download/"
               "v2checkpoints/vssmsmall_dp03_ckpt_epoch_238.pth")
        print(f"  Downloading weights via wget...")
        subprocess.run(f"wget -q '{url}' -O '{ckpt_path}'",
                       shell=True, check=True)
    return VSSM, ckpt_path


# ── VMamba Backbone Wrapper ────────────────────────────────────────────────────

class VMambaBackbone(nn.Module):
    """
    Wraps VMamba-Small backbone, OR falls back to ConvNeXt-Small if mamba-ssm unavailable.

    Both have identical channel dims [96, 192, 384, 768], so FPN decoder works unchanged.
    """

    OUT_CHANNELS = [96, 192, 384, 768]

    def __init__(self, pretrained: bool = True):
        super().__init__()

        # Try real VMamba-Small first
        try:
            from mamba_ssm import Mamba
            print("  ✅  mamba-ssm available — using real VMamba-Small")
            self.vssm = self._load_vmamba_official(pretrained)
            self.backbone_name = "VMamba-Small (real SSM)"
        except ImportError:
            print("  ⚠️  mamba-ssm not available — using ConvNeXt-Small fallback")
            print("      (for research paper: note this uses ConvNeXt, not Mamba SSM)")
            self.vssm = timm.create_model(
                "convnext_small", pretrained=pretrained,
                features_only=True, out_indices=(0, 1, 2, 3)
            )
            self.backbone_name = "ConvNeXt-Small (fallback)"

        self.out_channels = self.OUT_CHANNELS

    def _load_vmamba_official(self, pretrained: bool):
        """Load official VMamba-Small from HuggingFace or GitHub."""
        from huggingface_hub import hf_hub_download

        vmamba_src = Path("/content/VMamba")
        if not vmamba_src.exists():
            print("    Cloning VMamba...")
            subprocess.run(
                "git clone https://github.com/MzeroMiko/VMamba.git /content/VMamba --depth 1",
                shell=True, capture_output=True
            )

        vmamba_cls_path = str(vmamba_src / "classification")
        if vmamba_cls_path not in sys.path:
            sys.path.insert(0, vmamba_cls_path)

        subprocess.run("pip install -e /content/VMamba --quiet",
                      shell=True, capture_output=True)

        from models.vmamba import VSSM

        # Download weights
        ckpt_path = hf_hub_download(
            repo_id="MzeroMiko/VMamba",
            filename="vssmsmall_dp03_ckpt_epoch_238.pth",
            cache_dir="/content/vmamba_weights",
        )

        # Build model
        vssm = VSSM(
            patch_size=4, in_chans=3,
            depths=[2, 2, 15, 2], dims=[96, 192, 384, 768],
            ssm_d_state=16, ssm_ratio=2.0, ssm_rank_ratio=2.0,
            ssm_dt_rank="auto", ssm_act_layer="silu", ssm_conv=3,
            ssm_conv_bias=False, ssm_drop_rate=0.0, ssm_init="v0",
            forward_type="v2", mlp_ratio=0.0, mlp_act_layer="gelu",
            mlp_drop_rate=0.0, gmlp=False, patch_norm=True,
            norm_layer="ln", downsample_version="v3",
            patchembed_version="v2", use_checkpoint=False,
            posembed=False, imgsize=224,
        )

        if pretrained:
            state = torch.load(ckpt_path, map_location="cpu")
            if "model" in state:
                state = state["model"]
            elif "state_dict" in state:
                state = state["state_dict"]
            state = {k: v for k, v in state.items()
                     if not k.startswith(("head", "classifier", "norm"))}
            missing, unexpected = vssm.load_state_dict(state, strict=False)
            print(f"    Weights loaded  |  missing={len(missing)}  unexpected={len(unexpected)}")

        return vssm

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (B, 3, H, W)
        Returns:
            List of 4 feature tensors (B, C, H, W):
              [0]: (B,  96, H/4,  W/4)
              [1]: (B, 192, H/8,  W/8)
              [2]: (B, 384, H/16, W/16)
              [3]: (B, 768, H/32, W/32)
        """
        stage_outs = self.vssm.forward_features(x) if hasattr(self.vssm, 'forward_features') else self.vssm(x)

        # Check if output is already in (B, C, H, W) format (ConvNeXt) or (B, H, W, C) (VMamba)
        if isinstance(stage_outs, list):
            if stage_outs[0].ndim == 4 and stage_outs[0].shape[1] in [96, 192, 384, 768]:
                # Already (B, C, H, W) — ConvNeXt format
                return stage_outs
            else:
                # (B, H, W, C) — VMamba format, need to permute
                return [f.permute(0, 3, 1, 2).contiguous() for f in stage_outs]
        else:
            # Single output, reshape as needed
            if stage_outs.ndim == 4 and stage_outs.shape[1] in [96, 192, 384, 768]:
                return [stage_outs]
            return [stage_outs.permute(0, 3, 1, 2).contiguous()]

class ConvBNReLU(nn.Sequential):
    """Conv2d → BatchNorm2d → ReLU building block."""
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int = 1):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class FPNDecoder(nn.Module):
    """Top-down Feature Pyramid Network decoder."""
    def __init__(self, in_channels, fpn_channels: int = FPN_CHANNELS):
        super().__init__()
        self.lateral = nn.ModuleList(
            [nn.Conv2d(c, fpn_channels, 1) for c in in_channels])
        self.smooth = nn.ModuleList(
            [ConvBNReLU(fpn_channels, fpn_channels) for _ in in_channels])
        self.density_head = nn.Sequential(
            ConvBNReLU(fpn_channels, 128),
            ConvBNReLU(128, 64),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, features):
        f0, f1, f2, f3 = features
        p3 = self.smooth[3](self.lateral[3](f3))
        p2 = self.smooth[2](
            self.lateral[2](f2)
            + F.interpolate(p3, size=f2.shape[-2:], mode="bilinear", align_corners=False))
        p1 = self.smooth[1](
            self.lateral[1](f1)
            + F.interpolate(p2, size=f1.shape[-2:], mode="bilinear", align_corners=False))
        p0 = self.smooth[0](
            self.lateral[0](f0)
            + F.interpolate(p1, size=f0.shape[-2:], mode="bilinear", align_corners=False))
        if DENSITY_SCALE > 4:
            pool_factor = DENSITY_SCALE // 4
            p0 = F.avg_pool2d(p0, kernel_size=pool_factor, stride=pool_factor)
        return self.density_head(p0)


class VMambaCrowdCounter(nn.Module):
    """Full crowd counting model: VMamba-Small → FPN decoder → density head"""
    def __init__(self, pretrained: bool = True, fpn_channels: int = FPN_CHANNELS):
        super().__init__()
        self.backbone = VMambaBackbone(pretrained=pretrained)
        self.decoder  = FPNDecoder(self.backbone.out_channels, fpn_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.decoder(features)

    @staticmethod
    def count_from_density(density: torch.Tensor) -> torch.Tensor:
        """Sum density map over spatial dims to get crowd count."""
        return density.sum(dim=(1, 2, 3))


print("VMambaBackbone class defined (with ConvNeXt fallback).")
# ── Build model ───────────────────────────────────────────────────────────────
print("Building VMambaCrowdCounter...")
model = VMambaCrowdCounter(pretrained=True).to(DEVICE)
print(f"  Backbone: {model.backbone.backbone_name}")

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params    : {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")

print("\nForward-pass sanity check...")
model.eval()
with torch.no_grad():
    dummy     = torch.randn(1, 3, 256, 256, device=DEVICE)
    dummy_out = model(dummy)
print(f"  Input  : {list(dummy.shape)}")
print(f"  Output : {list(dummy_out.shape)}")
assert list(dummy_out.shape) == [1, 1, 32, 32]
print("✅  Sanity check passed: (1,3,256,256) → (1,1,32,32)")
model.train()

In [ ]:
# CELL 8

# ── Building blocks — IDENTICAL to working Swin notebook ─────────────────────

class ConvBNReLU(nn.Sequential):
    """Conv2d → BatchNorm2d → ReLU building block."""
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int = 1):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class FPNDecoder(nn.Module):
    """
    Top-down Feature Pyramid Network decoder.

    Takes 4 backbone feature maps (coarse → fine) and produces a single
    fused feature map at stride-4 (H/4, W/4), which is then pooled to
    stride-8 (H/8, W/8) when DENSITY_SCALE = 8.

    Implementation is IDENTICAL to the working Swin notebook.
    """

    def __init__(self, in_channels, fpn_channels: int = FPN_CHANNELS):
        super().__init__()
        # 1×1 lateral projections to fpn_channels for each stage
        self.lateral = nn.ModuleList(
            [nn.Conv2d(c, fpn_channels, 1) for c in in_channels])
        # 3×3 smooth convolutions after each merge
        self.smooth = nn.ModuleList(
            [ConvBNReLU(fpn_channels, fpn_channels) for _ in in_channels])
        # Density prediction head
        self.density_head = nn.Sequential(
            ConvBNReLU(fpn_channels, 128),
            ConvBNReLU(128, 64),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU(inplace=True),   # density cannot be negative
        )

    def forward(self, features):
        f0, f1, f2, f3 = features   # stride 4, 8, 16, 32

        # Top-down path: start from coarsest (f3) and merge upwards
        p3 = self.smooth[3](self.lateral[3](f3))
        p2 = self.smooth[2](
            self.lateral[2](f2)
            + F.interpolate(p3, size=f2.shape[-2:], mode="bilinear", align_corners=False))
        p1 = self.smooth[1](
            self.lateral[1](f1)
            + F.interpolate(p2, size=f1.shape[-2:], mode="bilinear", align_corners=False))
        p0 = self.smooth[0](
            self.lateral[0](f0)
            + F.interpolate(p1, size=f0.shape[-2:], mode="bilinear", align_corners=False))

        # Downsample from stride-4 → DENSITY_SCALE (stride-8)
        if DENSITY_SCALE > 4:
            pool_factor = DENSITY_SCALE // 4   # = 2 when DENSITY_SCALE=8
            p0 = F.avg_pool2d(p0, kernel_size=pool_factor, stride=pool_factor)

        return self.density_head(p0)


class VMambaCrowdCounter(nn.Module):
    """
    Full crowd counting model:
        VMamba-Small backbone  →  FPN decoder  →  density head

    Only the backbone changes vs. SwinCrowdCounter; the decoder and head
    are identical. This is the model submitted in the research paper.
    """

    def __init__(self, pretrained: bool = True, fpn_channels: int = FPN_CHANNELS):
        super().__init__()
        self.backbone = VMambaBackbone(pretrained=pretrained)
        self.decoder  = FPNDecoder(self.backbone.out_channels, fpn_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.decoder(features)

    @staticmethod
    def count_from_density(density: torch.Tensor) -> torch.Tensor:
        """Sum density map over spatial dims to get crowd count per image."""
        return density.sum(dim=(1, 2, 3))


# ── Build model ───────────────────────────────────────────────────────────────
print("Building VMambaCrowdCounter...")
model = VMambaCrowdCounter(pretrained=True).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params    : {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")

# ── Forward-pass sanity check ─────────────────────────────────────────────────
print("\nForward-pass sanity check...")
model.eval()
with torch.no_grad():
    dummy     = torch.randn(1, 3, 256, 256, device=DEVICE)
    dummy_out = model(dummy)
print(f"  Input  : {list(dummy.shape)}")
print(f"  Output : {list(dummy_out.shape)}")
assert list(dummy_out.shape) == [1, 1, 32, 32], (
    f"Expected output (1,1,32,32) but got {list(dummy_out.shape)}"
)
print("✅  Sanity check passed: (1,3,256,256) → (1,1,32,32)")
model.train()


In [ ]:
# CELL 9

class CrowdLoss(nn.Module):
    """
    MSE density loss + L1 count loss.

    Loss = MSE(pred_density, gt_density) + lambda_count × L1(pred_count, gt_count)

    MSE trains the spatial density distribution.
    L1 count trains the total integral (crowd count) explicitly.
    lambda_count=0.1 gives count loss ~10% weight — empirically optimal.
    """

    def __init__(self, lambda_count: float = LAMBDA_COUNT):
        super().__init__()
        self.mse          = nn.MSELoss()
        self.lambda_count = lambda_count

    def forward(self, pred: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:
        loss_density = self.mse(pred, gt)
        pred_count   = pred.sum(dim=(1, 2, 3))
        gt_count     = gt.sum(dim=(1, 2, 3))
        loss_count   = F.l1_loss(pred_count, gt_count)
        return loss_density + self.lambda_count * loss_count


def get_scheduler(optimizer, warmup_epochs: int = WARMUP_EPOCHS,
                  total_epochs: int = NUM_EPOCHS,
                  lr_min: float = LR_MIN,
                  lr_base: float = LEARNING_RATE):
    """Linear warmup (0 → lr_base over warmup_epochs) then cosine decay to lr_min."""
    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        cosine   = 0.5 * (1 + math.cos(math.pi * progress))
        return lr_min / lr_base + (1 - lr_min / lr_base) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


criterion = CrowdLoss(lambda_count=LAMBDA_COUNT)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_scheduler(optimizer)

print("✅  Loss, optimizer, scheduler ready")
print(f"   Loss : MSE(density) + {LAMBDA_COUNT} × L1(count)")
print(f"   LR   : {LEARNING_RATE}  (warmup {WARMUP_EPOCHS} epochs → cosine to {LR_MIN})")
print(f"   WD   : {WEIGHT_DECAY}")


In [ ]:
# CELL 10

def save_checkpoint(epoch: int, best_mae: float) -> None:
    """Save full training state to Google Drive. Called on every MAE improvement."""
    torch.save({
        "epoch"          : epoch,
        "model_state"    : model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_mae"       : best_mae,
    }, str(CHECKPOINT_PATH))
    print(f"  💾  Checkpoint saved  epoch={epoch+1}  MAE={best_mae:.2f}  → {CHECKPOINT_PATH.name}")


def load_checkpoint() -> tuple:
    """
    Load full training state from the checkpoint on Drive.
    Returns (start_epoch, best_mae).
    """
    if not CHECKPOINT_PATH.exists():
        print("  ℹ️   No checkpoint found — starting from epoch 1.")
        return 0, float("inf")

    ckpt = torch.load(str(CHECKPOINT_PATH), map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_mae    = ckpt["best_mae"]
    print(f"  ✅  Checkpoint loaded from {CHECKPOINT_PATH.name}")
    print(f"     Resuming at epoch {start_epoch + 1}  |  Best MAE so far: {best_mae:.2f}")
    return start_epoch, best_mae


print("✅  Checkpoint utilities ready")
print(f"   Save path: {CHECKPOINT_PATH}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LIVE DEMO — Upload Crowd Image/Video for Professor
# ═══════════════════════════════════════════════════════════════

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from google.colab.patches import cv2_imshow
from google.colab import files
from IPython.display import HTML, display
import base64
from io import BytesIO
from PIL import Image
import time

# ── Load Best Checkpoint ─────────────────────────────────────
print("Loading best model...")
ckpt = torch.load(str(CHECKPOINT_PATH), map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✅ Loaded checkpoint (MAE={ckpt['best_mae']:.2f})")

# ── Inference Function (Optimized) ───────────────────────────
normalize = transforms.Normalize(mean=IMG_MEAN, std=IMG_STD)

@torch.no_grad()
def predict_frame(frame_bgr):
    """
    Fast inference on a single frame.
    Returns: (density_map, count, inference_time_ms)
    """
    t0 = time.time()

    # Convert BGR→RGB
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    H, W = frame_rgb.shape[:2]

    # Pad to multiple of 32
    pad_H = (32 - H % 32) % 32
    pad_W = (32 - W % 32) % 32

    # To tensor + normalize
    img_t = torch.from_numpy(frame_rgb.transpose(2, 0, 1)).float() / 255.0
    img_t = normalize(img_t).unsqueeze(0).to(DEVICE)

    # Pad
    if pad_H > 0 or pad_W > 0:
        img_t = F.pad(img_t, (0, pad_W, 0, pad_H), mode='reflect')

    # Inference
    density = model(img_t)

    # Crop back
    dH = H // DENSITY_SCALE
    dW = W // DENSITY_SCALE
    density = density[0, 0, :dH, :dW].cpu().numpy()

    count = float(density.sum())
    inference_ms = (time.time() - t0) * 1000

    return density, count, inference_ms

# ── Visualization Function ──────────────────────────────────
def overlay_heatmap(frame, density, count, fps):
    """
    Overlay density heatmap + count + FPS on frame.
    Professional, non-crude visualization.
    """
    H, W = frame.shape[:2]

    # Resize density map to frame size
    density_resized = cv2.resize(density, (W, H))

    # Normalize density to 0-255 for colormap
    density_norm = np.clip(density_resized / (density_resized.max() + 1e-6), 0, 1)
    density_colored = cv2.applyColorMap(
        (density_norm * 255).astype(np.uint8),
        cv2.COLORMAP_JET
    )

    # Blend with original frame (50% opacity)
    overlay = cv2.addWeighted(frame, 0.5, density_colored, 0.5, 0)

    # Add professional UI elements
    # Top bar: dark background
    cv2.rectangle(overlay, (0, 0), (W, 80), (0, 0, 0), -1)
    cv2.rectangle(overlay, (0, 0), (W, 80), (255, 255, 255), 2)

    # Count (large, centered)
    count_text = f"Count: {int(count)}"
    font = cv2.FONT_HERSHEY_DUPLEX
    (tw, th), _ = cv2.getTextSize(count_text, font, 1.5, 3)
    cv2.putText(overlay, count_text,
                (W//2 - tw//2, 50),
                font, 1.5, (0, 255, 0), 3, cv2.LINE_AA)

    # FPS counter (top-right)
    fps_text = f"{fps:.1f} FPS"
    cv2.putText(overlay, fps_text, (W - 150, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

    # Model info (top-left)
    cv2.putText(overlay, "ConvNeXt-Small + FPN", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(overlay, f"MAE: {ckpt['best_mae']:.2f}", (10, 55),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1, cv2.LINE_AA)

    return overlay

# ═══════════════════════════════════════════════════════════════
#  CHOOSE ONE:
#  A) Image Upload  B) Video Upload  C) Webcam (see below)
# ═══════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────
#  OPTION A: Single Image Upload
# ──────────────────────────────────────────────────────────────
print("\n📤 Upload a crowd image (JPEG/PNG)...")
uploaded = files.upload()

for filename, data in uploaded.items():
    # Read image
    img = Image.open(BytesIO(data))
    frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

    # Predict
    density, count, inf_ms = predict_frame(frame)

    # Visualize
    result = overlay_heatmap(frame, density, count, fps=1000/inf_ms)

    # Display
    print(f"\n{'='*60}")
    print(f" RESULT: {int(count)} people detected")
    print(f" Inference time: {inf_ms:.1f} ms")
    print(f"{'='*60}\n")

    cv2_imshow(result)

    # Save result
    output_path = f"/content/drive/MyDrive/Metro-Crowd-Project/demo_{filename}"
    cv2.imwrite(output_path, result)
    print(f"✅ Saved result to Drive: demo_{filename}")

In [ ]:
# ──────────────────────────────────────────────────────────────
#  OPTION B: Video Upload (Process Every Frame)
# ──────────────────────────────────────────────────────────────

print("\n📤 Upload a crowd video (MP4/AVI)...")
uploaded = files.upload()

for filename, _ in uploaded.items():
    cap = cv2.VideoCapture(filename)

    # Get video properties
    fps_in = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Output video writer
    output_path = f"/content/drive/MyDrive/Metro-Crowd-Project/demo_output.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_in, (W, H))

    print(f"Processing {total_frames} frames at {fps_in} FPS...")

    frame_times = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Predict
        density, count, inf_ms = predict_frame(frame)
        frame_times.append(inf_ms)

        # Compute rolling average FPS
        avg_fps = 1000 / np.mean(frame_times[-30:]) if len(frame_times) > 0 else 0

        # Overlay
        result = overlay_heatmap(frame, density, count, avg_fps)

        # Write frame
        out.write(result)

        frame_idx += 1
        if frame_idx % 30 == 0:
            print(f"  [{frame_idx}/{total_frames}] FPS: {avg_fps:.1f}, Count: {int(count)}")

    cap.release()
    out.release()

    print(f"\n✅ Output video saved: {output_path}")
    print(f"   Avg inference: {np.mean(frame_times):.1f} ms/frame")
    print(f"   Avg FPS: {1000/np.mean(frame_times):.1f}")

In [ ]:
# ──────────────────────────────────────────────────────────────
#  OPTION C: Webcam Capture (JavaScript-based, Colab workaround)
# ──────────────────────────────────────────────────────────────

from IPython.display import display, Javascript
from google.colab.output import eval_js

def take_photo(quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture Frame';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize for performance
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for capture
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    return data

# Capture and predict
print("Click 'Capture Frame' when ready...")
img_data = take_photo()

# Decode base64 → image
img_bytes = base64.b64decode(img_data.split(',')[1])
img = Image.open(BytesIO(img_bytes))
frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

# Predict
density, count, inf_ms = predict_frame(frame)
result = overlay_heatmap(frame, density, count, 1000/inf_ms)

print(f"\n📊 Detected: {int(count)} people ({inf_ms:.1f} ms)")
cv2_imshow(result)

In [ ]:
# CELL 11

def train_one_epoch(epoch: int) -> float:
    model.train()
    total_loss = 0.0

    for i, (imgs, densities, _) in enumerate(train_loader):
        imgs      = imgs.to(DEVICE, non_blocking=True)
        densities = densities.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        pred = model(imgs)

        # Shape guard: pred should already match densities for 256×256 crops
        if pred.shape[-2:] != densities.shape[-2:]:
            pred = F.interpolate(pred, size=densities.shape[-2:],
                                 mode="bilinear", align_corners=False)

        loss = criterion(pred, densities)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        if (i + 1) % 20 == 0:
            print(f"    [Ep {epoch:03d}  batch {i+1:03d}/{len(train_loader):03d}]"
                  f"  loss={loss.item():.5f}")

    return total_loss / len(train_loader)


@torch.no_grad()
def validate() -> tuple:
    """
    Full-image validation on the official ShanghaiTech test set.

    Strategy:
      1. Pad each image to the nearest multiple of 32 (VMamba patch-embed safe).
      2. Run the model.
      3. Crop prediction back to (H // DENSITY_SCALE, W // DENSITY_SCALE).
      4. Compare integral (count) to ground-truth count.
    """
    model.eval()
    mae_sum = mse_sum = 0.0
    n = 0

    for imgs, densities, _ in test_loader:
        imgs      = imgs.to(DEVICE, non_blocking=True)
        densities = densities.to(DEVICE, non_blocking=True)

        B, C, H, W = imgs.shape

        # Pad to multiple of 32 so VMamba patch-embed windows tile evenly
        pad_H = (32 - H % 32) % 32
        pad_W = (32 - W % 32) % 32
        if pad_H > 0 or pad_W > 0:
            imgs_padded = F.pad(imgs, (0, pad_W, 0, pad_H), mode="reflect")
        else:
            imgs_padded = imgs

        pred = model(imgs_padded)

        # Crop prediction back to the expected density-map resolution
        dH = H // DENSITY_SCALE
        dW = W // DENSITY_SCALE
        pred = pred[:, :, :dH, :dW]

        # Shape guard (safety net)
        if pred.shape[-2:] != densities.shape[-2:]:
            pred = F.interpolate(pred, size=densities.shape[-2:],
                                 mode="bilinear", align_corners=False)

        pred_cnt = pred.sum(dim=(1, 2, 3)).cpu().numpy()
        gt_cnt   = densities.sum(dim=(1, 2, 3)).cpu().numpy()

        mae_sum += float(np.abs(pred_cnt - gt_cnt).sum())
        mse_sum += float(((pred_cnt - gt_cnt) ** 2).sum())
        n       += len(imgs)

    mae  = mae_sum / n
    rmse = math.sqrt(mse_sum / n)
    return mae, rmse


print("✅  train_one_epoch() and validate() defined")


In [ ]:
# CELL 12

# ════════════════════════════════════════════════════════════════
#  SET THIS FLAG BEFORE RUNNING:
#  False = start fresh from epoch 1
#  True  = resume from the last saved checkpoint on Drive
# ════════════════════════════════════════════════════════════════
RESUME = False

# ── Initialise state ──────────────────────────────────────────
start_epoch = 0
best_mae    = float("inf")
history     = {"epoch": [], "lr": [], "train_loss": [], "val_mae": [], "val_rmse": []}

if RESUME and CHECKPOINT_PATH.exists():
    start_epoch, best_mae = load_checkpoint()
else:
    if RESUME:
        print("⚠️  RESUME=True but no checkpoint found — starting from epoch 1.")
    else:
        print("Starting fresh (RESUME=False).")

print(f"\n{'═'*60}")
print(f" ConvNeXt Crowd Counter — Training  ({DATASET_PART})")
print(f" Epochs  : {start_epoch+1} → {NUM_EPOCHS}")
print(f" Device  : {DEVICE}")
print(f" Best MAE: {best_mae:.2f}  (from checkpoint)" if best_mae < 1e9 else f" Best MAE: N/A")
print(f"{'═'*60}\n")

# ── Training loop ─────────────────────────────────────────────
t_total = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    t_ep = time.time()
    current_lr = scheduler.get_last_lr()[0]
    print(f"\n{'─'*55}")
    print(f"EPOCH {epoch+1:3d}/{NUM_EPOCHS}   LR={current_lr:.2e}")

    train_loss        = train_one_epoch(epoch + 1)
    val_mae, val_rmse = validate()
    scheduler.step()

    history["epoch"     ].append(epoch + 1)
    history["lr"        ].append(current_lr)
    history["train_loss"].append(train_loss)
    history["val_mae"   ].append(val_mae)
    history["val_rmse"  ].append(val_rmse)

    ep_time = time.time() - t_ep
    print(f"  Loss={train_loss:.5f}   MAE={val_mae:.2f}   RMSE={val_rmse:.2f}"
          f"   time={ep_time:.0f}s")

    if val_mae < best_mae:
        best_mae = val_mae
        save_checkpoint(epoch, best_mae)
        print(f"  🏆  New best MAE = {best_mae:.2f}")

total_time = (time.time() - t_total) / 60
print(f"\n{'═'*60}")
print(f" Training complete  |  Best MAE = {best_mae:.2f}")
print(f" Total time: {total_time:.1f} minutes")
print(f"{'═'*60}")


In [ ]:
# CELL 13

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"ConvNeXt-Small + FPN — ShanghaiTech {DATASET_PART}", fontsize=13, fontweight="bold")

axes[0].plot(history["epoch"], history["train_loss"], color="steelblue", lw=2)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE + 0.1×L1_count")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["epoch"], history["val_mae"], color="tomato", lw=2)
axes[1].axhline(41.45, color="gray", lw=1, linestyle="--", label="Swin-S baseline (41.45)")
axes[1].set_title("Validation MAE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history["epoch"], history["val_rmse"], color="darkorange", lw=2)
axes[2].axhline(70.0, color="gray", lw=1, linestyle="--", label="Swin-S baseline (~70)")
axes[2].set_title("Validation RMSE")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("RMSE")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CURVES_PATH), dpi=150, bbox_inches="tight")
plt.show()
print(f"✅  Curves saved → {CURVES_PATH}")


In [ ]:
# CELL 14

# Load best model
print("Loading best checkpoint for final evaluation...")
ckpt = torch.load(str(CHECKPOINT_PATH), map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"  Loaded epoch {ckpt['epoch']+1}  |  best MAE during training: {ckpt['best_mae']:.2f}")

# Full test-set evaluation
print("\nRunning full test-set evaluation...")
test_mae, test_rmse = validate()

print(f"\n{'═'*55}")
print(f" FINAL TEST RESULTS — ShanghaiTech {DATASET_PART}")
print(f"{'═'*55}")
print(f"  ConvNeXt-Small + FPN  |  MAE = {test_mae:.2f}   RMSE = {test_rmse:.2f}")
print(f"{'─'*55}")
print(f"  Comparison table:")
print(f"  {'Model':<30} {'MAE':>8} {'RMSE':>8}")
print(f"  {'─'*48}")
print(f"  {'ConvNeXt-Small + FPN (ours)':<30} {test_mae:>8.2f} {test_rmse:>8.2f}")
print(f"  {'Swin-Small + FPN (baseline)':<30} {'41.45':>8} {'~70':>8}")
print(f"  {'CSRNet / VGG-16 (SOTA 2018)':<30} {'10.6':>8} {'16.0':>8}")
print(f"  {'─'*48}")

improvement_mae  = 41.45 - test_mae
improvement_rmse = 70.0  - test_rmse
if improvement_mae > 0:
    print(f"\n  ✅  ConvNeXt beats Swin baseline by {improvement_mae:.2f} MAE  ({improvement_mae/41.45*100:.1f}% improvement)")
else:
    print(f"\n  ⚠️  ConvNeXt does not yet beat Swin baseline (gap: {-improvement_mae:.2f} MAE)")
    print(f"     → Training may need more epochs or hyperparameter tuning.")
print(f"{'═'*55}")


In [ ]:
# CELL 15

def denormalize(tensor: torch.Tensor) -> np.ndarray:
    """Convert normalised image tensor back to uint8 RGB array."""
    mean = torch.tensor(IMG_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMG_STD ).view(3, 1, 1)
    img  = (tensor.cpu() * std + mean).clamp(0, 1).numpy()
    return (img.transpose(1, 2, 0) * 255).astype(np.uint8)


@torch.no_grad()
def visualise_predictions(n_images: int = 4) -> None:
    model.eval()
    fig, axes = plt.subplots(n_images, 3, figsize=(15, 5 * n_images))
    fig.suptitle(f"ConvNeXt-Small + FPN — Predictions  (ShanghaiTech {DATASET_PART})",
                 fontsize=14, fontweight="bold", y=1.01)
    for ax, title in zip(axes[0], ["Input Image", "GT Density Map", "Predicted Density Map"]):
        ax.set_title(title, fontsize=11, fontweight="bold")

    sample_idx = 0
    for imgs, densities, paths in test_loader:
        if sample_idx >= n_images:
            break

        B, C, H, W = imgs.shape
        pad_H = (32 - H % 32) % 32
        pad_W = (32 - W % 32) % 32
        imgs_padded = F.pad(imgs, (0, pad_W, 0, pad_H), mode="reflect") if (pad_H or pad_W) else imgs
        imgs_padded = imgs_padded.to(DEVICE)
        densities   = densities.to(DEVICE)

        pred = model(imgs_padded)[:, :, :H//DENSITY_SCALE, :W//DENSITY_SCALE]
        if pred.shape[-2:] != densities.shape[-2:]:
            pred = F.interpolate(pred, size=densities.shape[-2:],
                                 mode="bilinear", align_corners=False)

        for b in range(B):
            if sample_idx >= n_images:
                break

            img_np  = denormalize(imgs[b])
            gt_np   = densities[b, 0].cpu().numpy()
            pred_np = pred[b, 0].cpu().numpy()
            gt_cnt  = gt_np.sum()
            pr_cnt  = pred_np.sum()

            axes[sample_idx][0].imshow(img_np)
            axes[sample_idx][0].axis("off")
            axes[sample_idx][0].set_xlabel(Path(paths[b]).name, fontsize=8)

            axes[sample_idx][1].imshow(gt_np, cmap="jet")
            axes[sample_idx][1].axis("off")
            axes[sample_idx][1].set_xlabel(f"GT count: {gt_cnt:.1f}", fontsize=10)

            axes[sample_idx][2].imshow(pred_np, cmap="jet")
            axes[sample_idx][2].axis("off")
            axes[sample_idx][2].set_xlabel(f"Pred count: {pr_cnt:.1f}", fontsize=10)

            sample_idx += 1

    plt.tight_layout()
    vis_path = CHECKPOINT_DIR / f"vmamba_{DATASET_PART}_predictions.png"
    plt.savefig(str(vis_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅  Visualisation saved → {vis_path}")


visualise_predictions(n_images=4)


In [ ]:
# CELL 16

@torch.no_grad()
def predict_single_image(image_path: str) -> tuple:
    """
    Run inference on a single image.

    Args:
        image_path: path to a JPEG or PNG crowd image.
    Returns:
        (density_map: np.ndarray, count: float)
    """
    model.eval()
    norm = transforms.Normalize(mean=IMG_MEAN, std=IMG_STD)

    img     = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    H, W    = img.shape[:2]

    # To tensor + normalise
    t = norm(torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0)

    # Pad to multiple of 32
    pad_H = (32 - H % 32) % 32
    pad_W = (32 - W % 32) % 32
    t_padded = F.pad(t.unsqueeze(0), (0, pad_W, 0, pad_H), mode="reflect")

    t_padded  = t_padded.to(DEVICE)
    pred      = model(t_padded)                         # (1, 1, (H+pad_H)/8, ...)
    density   = pred[0, 0, :H//DENSITY_SCALE, :W//DENSITY_SCALE].cpu().numpy()
    count     = float(density.sum())

    # Visualise
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].imshow(img)
    axes[0].set_title(f"Input: {Path(image_path).name}")
    axes[0].axis("off")
    im = axes[1].imshow(density, cmap="jet")
    axes[1].set_title(f"Predicted density map  |  Estimated count = {count:.1f}")
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    print(f"✅  Estimated crowd count: {count:.1f}")
    return density, count


# Demo: run on a random test image
sample_img_path = str(next(Path(TEST_IMG_PATH).glob("*.jpg")))
density_map, estimated_count = predict_single_image(sample_img_path)


In [ ]:
import numpy as np

def get_stats(cache_path):
    with open(cache_path, 'rb') as f:
        cache = pickle.load(f)
    counts = [d.sum() for d in cache.values()]
    return np.mean(counts), np.std(counts), len(counts)

train_mu, train_std, train_n = get_stats(TRAIN_CACHE)
test_mu, test_std, test_n = get_stats(TEST_CACHE)

print(f"--- Dataset Statistics ({DATASET_PART}) ---")
print(f"Train: n={train_n}, mean={train_mu:.2f}, std={train_std:.2f}")
print(f"Test:  n={test_n}, mean={test_mu:.2f}, std={test_std:.2f}")

# Check if the difficulty level is significantly different
ratio = test_mu / train_mu
print(f"\nTest/Train Mean Ratio: {ratio:.2f}")
if 0.8 < ratio < 1.2:
    print("\nResults Note: The train and test distributions are very similar. In Part B, images are taken from the same cameras/angles, so ConvNeXt-Small with an FPN head can achieve extremely high accuracy (single-digit MAE) because it learns the perspective and background of these specific locations very well.")